In [1]:
import numpy as np
import pandas as pd   
import statsmodels.api as sm 
import matplotlib.pyplot as plt 
import numpy as np  


In [2]:
sample_data = pd.read_pickle('../DataSummary/Aggregate_CPI_inflation_Quarterly.pkl')

In [3]:
df = sample_data

In [1]:
.865019**2

0.748257870361

In [4]:
df.describe()

,Year,Quarter,Price index,Inflation,Forecasted inflation,Inflation shock,Inflation_lag_1,Inflation_lag_2,Forecasted_inflation_lag_1
count,210.000000,210.000000,210.000000,210.000000,210.000000,210.000000,210.000000,210.000000,210.000000
mean,1996.247619,2.509524,156.233495,0.980799,0.833311,0.147488,0.983080,0.987732,0.832562
std,15.193036,1.120665,70.232253,0.865019,0.502730,0.666906,0.865241,0.865877,0.502725
min,1970.000000,1.000000,39.200000,-3.416988,0.104789,-4.011699,-3.416988,-3.416988,0.104789
25%,1983.000000,2.000000,100.650000,0.524367,0.473978,-0.169851,0.524367,0.524367,0.473978
50%,1996.000000,3.000000,158.400000,0.781601,0.622432,0.100099,0.781601,0.786865,0.622432
75%,2009.000000,3.750000,217.351500,1.269274,1.016893,0.443092,1.279417,1.282804,1.016893
max,2022.000000,4.000000,298.990000,4.161248,2.456609,2.172944,4.161248,4.161248,2.456609


In [4]:
df.describe()

,Inflation,Inflation_lag_1,Inflation_lag_2,SPF,SPF_lag_1,SPF_shock
count,215.000000,215.000000,215.000000,215.000000,215.000000,215.000000
mean,0.991782,0.994471,0.998238,0.832024,0.831550,0.159759
std,0.858044,0.858429,0.858531,0.496899,0.496868,0.664175
min,-3.416988,-3.416988,-3.416988,0.104789,0.104789,-4.011699
25%,0.532850,0.532850,0.532850,0.474794,0.474794,-0.166923
50%,0.793651,0.793651,0.796687,0.629643,0.629643,0.110065
75%,1.291325,1.299207,1.304400,1.014998,1.014998,0.461650
max,4.161248,4.161248,4.161248,2.456609,2.456609,2.172944


In [5]:
L = 12

# Regression 1
X1 = sm.add_constant(df[['Inflation_lag_1', 'SPF']])
model1 = sm.OLS(df['Inflation'], X1).fit(
    cov_type='HAC',
    cov_kwds={'maxlags': L}
)

# Regression 2
X2 = sm.add_constant(df[['Inflation_lag_1', 'Inflation_lag_2', 'SPF']])
model2 = sm.OLS(df['Inflation'], X2).fit(
    cov_type='HAC',
    cov_kwds={'maxlags': L}
)

# Regression 3
X3 = sm.add_constant(df[['Inflation_lag_1', 'Inflation_lag_2',
                          'SPF', 'SPF_lag_1']])
model3 = sm.OLS(df['Inflation'], X3).fit(
    cov_type='HAC',
    cov_kwds={'maxlags': L}
)


In [6]:
print(model1.summary())

                            OLS Regression Results                            
Dep. Variable:              Inflation   R-squared:                       0.450
Model:                            OLS   Adj. R-squared:                  0.445
Method:                 Least Squares   F-statistic:                     43.57
Date:                Mon, 11 May 2026   Prob (F-statistic):           1.41e-16
Time:                        11:07:32   Log-Likelihood:                -207.38
No. Observations:                 215   AIC:                             420.8
Df Residuals:                     212   BIC:                             430.9
Df Model:                           2                                         
Covariance Type:                  HAC                                         
                      coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------------------------------------
const               0.0824      0.086     

In [7]:
print(model2.summary())

                            OLS Regression Results                            
Dep. Variable:              Inflation   R-squared:                       0.453
Model:                            OLS   Adj. R-squared:                  0.445
Method:                 Least Squares   F-statistic:                     30.95
Date:                Mon, 11 May 2026   Prob (F-statistic):           1.27e-16
Time:                        11:08:52   Log-Likelihood:                -206.81
No. Observations:                 215   AIC:                             421.6
Df Residuals:                     211   BIC:                             435.1
Df Model:                           3                                         
Covariance Type:                  HAC                                         
                      coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------------------------------------
const               0.0897      0.086     

In [8]:
print(model3.summary())

                            OLS Regression Results                            
Dep. Variable:              Inflation   R-squared:                       0.454
Model:                            OLS   Adj. R-squared:                  0.443
Method:                 Least Squares   F-statistic:                     22.64
Date:                Mon, 11 May 2026   Prob (F-statistic):           1.45e-15
Time:                        11:08:58   Log-Likelihood:                -206.66
No. Observations:                 215   AIC:                             423.3
Df Residuals:                     210   BIC:                             440.2
Df Model:                           4                                         
Covariance Type:                  HAC                                         
                      coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------------------------------------
const               0.0856      0.086     

Automate reporting OLS results.

In [1]:
"""
Run the four mean-process specifications and export results to markdown.

Models:
  Constant : pi_{t+1} = SPF_t + mu_{t+1}                       (no estimated params)
  ARX(1,1) : pi_{t+1} = c + rho1*pi_t + phi1*SPF_t + mu
  ARX(2,1) : pi_{t+1} = c + rho1*pi_t + rho2*pi_{t-1} + phi1*SPF_t + mu
  ARX(2,2) : pi_{t+1} = c + rho1*pi_t + rho2*pi_{t-1} + phi1*SPF_t + phi2*SPF_{t-1} + mu

For each model we report:
  - parameter estimates with HAC(12) standard errors below in parentheses
  - log-likelihood (under Gaussian residuals)
  - AIC, BIC
  - skewness and excess kurtosis of residuals
"""

import numpy as np
import pandas as pd
import statsmodels.api as sm
from scipy import stats

# ---------------------------------------------------------------------------
# Data
# ---------------------------------------------------------------------------
df = pd.read_pickle('../DataSummary/Aggregate_CPI_inflation_Quarterly.pkl')
L = 12  # HAC lags

# ---------------------------------------------------------------------------
# Helpers
# ---------------------------------------------------------------------------
def diagnostics_constant(resid):
    """Diagnostics for the Constant model (no estimated coefficients).

    Log-likelihood under residuals ~ N(0, sigma^2) with sigma^2 by MLE.
    Only one parameter (sigma^2) is estimated, so k = 1 for AIC/BIC.
    """
    resid = np.asarray(resid)
    n = len(resid)
    sigma2 = np.mean(resid ** 2)
    llf = -0.5 * n * (np.log(2 * np.pi) + np.log(sigma2) + 1)
    k = 1
    aic = 2 * k - 2 * llf
    bic = k * np.log(n) - 2 * llf
    return dict(n=n, llf=llf, aic=aic, bic=bic,
                skew=stats.skew(resid, bias=False), kurt=stats.kurtosis(resid, bias=False))


def diagnostics_ols(model):
    """Diagnostics for an OLS model: use statsmodels' built-in llf/aic/bic.

    statsmodels computes llf under the Gaussian assumption and counts all
    regression coefficients plus sigma^2 in AIC/BIC.
    """
    resid = np.asarray(model.resid)
    return dict(n=int(model.nobs), llf=model.llf, aic=model.aic, bic=model.bic,
                skew=stats.skew(resid, bias=False), kurt=stats.kurtosis(resid, bias=False))


def render_equation(lhs, params, ses, names, decimals=4):
    r"""Render an equation with standard errors below each estimate.

    Uses a plain ``array`` environment (portable across LaTeX, MathJax, and
    Typst) rather than ``\phantom``/``\underset``, which Typst doesn't accept.

    Layout::

        lhs = b0   + b1*x1   + b2*x2  ...
              (s0)   (s1)      (s2)
    """
    # Build a list of "term" strings: ["b0", "+ b1 x1", "+ b2 x2", ...]
    terms = []
    for i, (b, name) in enumerate(zip(params, names)):
        val = f"{abs(b):.{decimals}f}"
        if i == 0:
            sign = "-" if b < 0 else ""
        else:
            sign = "-" if b < 0 else "+"
        if name == "1":  # intercept
            body = val
        else:
            body = f"{val}\\,{name}"
        terms.append(f"{sign}\\,{body}" if sign else body)

    # Build the array: one column per term, column alignment 'c'.
    ncol = len(terms)
    col_spec = "c" * ncol
    coef_row = " & ".join(terms)
    se_row = " & ".join(f"({se:.3f})" for se in ses)

    # Put LHS in the first column of the outer array and the inline
    # coefficient/SE block in the second. Use plain 'rl' (no @{...} spacing
    # trick, which some LaTeX→Typst converters reject).
    return (
        "$$\n"
        "\\begin{array}{rl}\n"
        f"{lhs} = & \\begin{{array}}{{{col_spec}}}\n"
        f"{coef_row} \\\\\n"
        f"{se_row}\n"
        "\\end{array}\n"
        "\\end{array}\n"
        "$$\n"
    )


def model_to_markdown(label, equation_lhs, params, ses, names, diag, dist_note):
    """Render one model block as markdown."""
    eq = render_equation(equation_lhs, params, ses, names)
    diag_lines = (
        # f"- Distribution for log-likelihood: **{dist_note}**\n"
        f"- Log-likelihood: **{diag['llf']:.3f}**\n"
        f"- AIC: **{diag['aic']:.3f}**, BIC: **{diag['bic']:.3f}**\n"
        f"- Residual skewness: **{diag['skew']:.3f}**, "
        f"excess kurtosis: **{diag['kurt']:.3f}**\n"
        # f"- N = {diag['n']}\n"
    )
    return f"**{label}**:\n\n{eq}\n{diag_lines}\n"


# ---------------------------------------------------------------------------
# Model 0: Constant (no estimated parameters)
# ---------------------------------------------------------------------------
resid0 = (df['Inflation'] - df['SPF']).dropna()
diag0 = diagnostics_constant(resid0)

# ---------------------------------------------------------------------------
# Model 1: ARX(1,1)
# ---------------------------------------------------------------------------
X1 = sm.add_constant(df[['Inflation_lag_1', 'SPF']])
model1 = sm.OLS(df['Inflation'], X1, missing='drop').fit(
    cov_type='HAC', cov_kwds={'maxlags': L}
)
diag1 = diagnostics_ols(model1)

# ---------------------------------------------------------------------------
# Model 2: ARX(2,1)
# ---------------------------------------------------------------------------
X2 = sm.add_constant(df[['Inflation_lag_1', 'Inflation_lag_2', 'SPF']])
model2 = sm.OLS(df['Inflation'], X2, missing='drop').fit(
    cov_type='HAC', cov_kwds={'maxlags': L}
)
diag2 = diagnostics_ols(model2)

# ---------------------------------------------------------------------------
# Model 3: ARX(2,2)
# ---------------------------------------------------------------------------
X3 = sm.add_constant(df[['Inflation_lag_1', 'Inflation_lag_2',
                          'SPF', 'SPF_lag_1']])
model3 = sm.OLS(df['Inflation'], X3, missing='drop').fit(
    cov_type='HAC', cov_kwds={'maxlags': L}
)
diag3 = diagnostics_ols(model3)


# ---------------------------------------------------------------------------
# Build markdown
# ---------------------------------------------------------------------------
header = (
    "# OLS Results\n\n"
    "Estimation results of four mean models with HAC standard errors "
    "in parentheses below each "
    "estimate. (The "
    "log-likelihood is computed under the Gaussian assumption.)\n\n"
)

# --- Constant model block (special: no estimated coefficients) ---
constant_block = (
    "**Constant**:\n\n"
    "$$\n"
    "\\hat{\\pi}_{t+1} = SPF_t\n"
    "$$\n\n"
    "No estimated coefficients. Residuals $\\mu_{t+1} = \\pi_{t+1} - SPF_t$.\n\n"
    # f"- Distribution for log-likelihood: **Gaussian ($\\mu \\sim \\mathcal{{N}}(0,\\sigma^2)$, $\\sigma^2$ estimated by MLE)**\n"
    f"- Log-likelihood: **{diag0['llf']:.3f}**\n"
    f"- AIC: **{diag0['aic']:.3f}**, BIC: **{diag0['bic']:.3f}**\n"
    f"- Residual skewness: **{diag0['skew']:.3f}**, "
    f"excess kurtosis: **{diag0['kurt']:.3f}**\n\n" 
)

# --- ARX blocks ---
arx11_block = model_to_markdown(
    label="ARX(1,1)",
    equation_lhs=r"\hat{\pi}_{t+1}",
    params=model1.params.values,
    ses=model1.bse.values,
    names=["1", r"\pi_t", r"SPF_t"],
    diag=diag1,
    dist_note=r"Gaussian ($\mu \sim \mathcal{N}(0,\sigma^2)$)",
)

arx21_block = model_to_markdown(
    label="ARX(2,1)",
    equation_lhs=r"\hat{\pi}_{t+1}",
    params=model2.params.values,
    ses=model2.bse.values,
    names=["1", r"\pi_t", r"\pi_{t-1}", r"SPF_t"],
    diag=diag2,
    dist_note=r"Gaussian ($\mu \sim \mathcal{N}(0,\sigma^2)$)",
)

arx22_block = model_to_markdown(
    label="ARX(2,2)",
    equation_lhs=r"\hat{\pi}_{t+1}",
    params=model3.params.values,
    ses=model3.bse.values,
    names=["1", r"\pi_t", r"\pi_{t-1}", r"SPF_t", r"SPF_{t-1}"],
    diag=diag3,
    dist_note=r"Gaussian ($\mu \sim \mathcal{N}(0,\sigma^2)$)",
)

markdown = header + constant_block + arx11_block + arx21_block + arx22_block

# ---------------------------------------------------------------------------
# Save
# ---------------------------------------------------------------------------
out_path = 'OLS_Results.md'
with open(out_path, 'w') as f:
    f.write(markdown)

print(f"Wrote {out_path}")

Wrote OLS_Results.md
